In [1]:
import os
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()


@dataclass(frozen=True)
class Provider:
    """One provider described as pure DATA (same design as Notebook 1)."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider("OpenAI",     "OPENAI_API_KEY",     False, None,                              "gpt-4o-mini"),
    Provider("Groq",       "GROQ_API_KEY",       True,  "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return provider
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set. Add one of {expected} to your .env file.")


def build_client(provider: Provider):
    from openai import OpenAI

    api_key = os.environ[provider.env_var]
    if provider.base_url is None:
        return OpenAI(api_key=api_key)
    return OpenAI(api_key=api_key, base_url=provider.base_url)


def have_any_key() -> bool:
    return any(os.environ.get(p.env_var) for p in PROVIDERS)



def llm_reply(prompt: str, *, max_tokens: int = 400) -> str:
    """Send one user prompt; return the assistant's text."""
    provider = select_provider()
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return result.choices[0].message.content

# Zero shot COT

In [2]:
ZERO_SHOT_COT_SUFFIX = "\n\nLet's think step by step."

def zero_shot_cot_prompt(question: str) -> str:
    """This function takes a question and returns a prompt for the zero-shot COT model."""
    return f"Question: {question}{ZERO_SHOT_COT_SUFFIX}"

In [3]:
QUESTION = (
    "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
    "Each can has 3 tennis balls. How many tennis balls does he have now?"
)

In [4]:
print(zero_shot_cot_prompt(QUESTION))

Question: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?

Let's think step by step.


In [5]:
reply = llm_reply(zero_shot_cot_prompt(QUESTION))

print(reply)

To find out how many tennis balls Roger has after buying more, we can follow these steps:

1. **Start with the initial number of tennis balls**: Roger starts with 5 tennis balls.

2. **Determine how many tennis balls he buys**: Roger buys 2 cans of tennis balls, and each can contains 3 tennis balls. To find out how many tennis balls he buys, we multiply the number of cans by the number of tennis balls in each can:
   \[
   2 \text{ cans} \times 3 \text{ tennis balls per can} = 6 \text{ tennis balls}
   \]

3. **Add the tennis balls he bought to his original amount**: Now we add the 6 tennis balls to the 5 tennis balls he already has:
   \[
   5 \text{ (original tennis balls)} + 6 \text{ (new tennis balls)} = 11 \text{ tennis balls}
   \]

So, Roger now has a total of **11 tennis balls**.


# Few Shot COT

In [6]:
FEW_SHOT_EXAMPLES = """
Q: There are 15 trees in the grove. Grove workers will plant 6 trees today. How many trees will be in the grove?
A: There are 15 trees to start. After planting 6 more, there are 15 + 6 = 21 trees.
Final answer: 21

Q: Leah had 32 chocolates and her sister had 42. They ate 35. How many pieces do they have left in total?
A: Leah and her sister had 32 + 42 = 74 chocolates together. After eating 35, they have 74 - 35 = 39 left.
Final answer: 39
""".strip()

In [7]:
def few_shot_cot_prompt(question: str) -> str:
    """This function takes a question and returns a prompt for the few-shot COT model."""
    return f"""Solve each question by reasoning step by step.
    {FEW_SHOT_EXAMPLES}
    
    Question: {question}
    Answer:
    """

In [8]:
reply = llm_reply(few_shot_cot_prompt(QUESTION))

print(reply)

Let's break down the problem step by step:

1. Roger starts with 5 tennis balls.
2. He buys 2 cans of tennis balls.
3. Each can contains 3 tennis balls, so the total number of tennis balls in the cans is:
   \( 2 \, \text{cans} \times 3 \, \text{tennis balls per can} = 6 \, \text{tennis balls} \).
4. Now, we add the tennis balls he had initially to the tennis balls he bought:
   \( 5 \, \text{tennis balls} + 6 \, \text{tennis balls} = 11 \, \text{tennis balls} \).

Final answer: 11


In [9]:
TRICKY = (
    "A juggler can juggle 16 balls. Half of the balls are golf balls, and "
    "half of the golf balls are blue. How many blue golf balls are there?"
)

In [10]:
reply = llm_reply(zero_shot_cot_prompt(TRICKY))

print(reply)

Let's break the problem down step by step.

1. **Total number of balls**: The juggler can juggle 16 balls.

2. **Half of the balls are golf balls**: Since half of the 16 balls are golf balls, we calculate:
   \[
   \text{Number of golf balls} = \frac{16}{2} = 8
   \]

3. **Half of the golf balls are blue**: Now, we want to find out how many of the golf balls are blue. Since half of the 8 golf balls are blue, we calculate:
   \[
   \text{Number of blue golf balls} = \frac{8}{2} = 4
   \]

Therefore, the number of blue golf balls is **4**.


In [11]:
reply = llm_reply(few_shot_cot_prompt(TRICKY))

print(reply)

To solve the problem step by step:

1. We know that the juggler can juggle a total of 16 balls.
2. It is stated that half of the balls are golf balls. Therefore, the number of golf balls is:
   \[
   \text{Golf balls} = \frac{16}{2} = 8
   \]
3. Next, it is mentioned that half of the golf balls are blue. So, we need to find out how many blue golf balls there are:
   \[
   \text{Blue golf balls} = \frac{8}{2} = 4
   \]

Final answer: 4
